# Model Training (Final Run)

Train the character-level GPT using the **best hyperparameters** from [4. Hyperparameter Tuning.ipynb](4.%20%20%20Hyperparameter%20Tuning.ipynb).

This notebook is organized as a linear pipeline — run each section in order:

| Step | Section | What it does |
|------|---------|--------------|
| 1 | Setup | Imports, paths, device |
| 2 | Config | Load sweep-best hyperparameters |
| 3 | Data | Token IDs + batch helper |
| 4 | Model | GPT architecture |
| 5 | LR schedule | Cosine warmup + decay |
| 6 | Training helpers | Optimizer, eval, checkpoint |
| 7 | Training loop | The `train_final_run` function |
| 8 | **Run training** | W&B init → train → restore best weights |
| 9 | Visualize | Loss / perplexity / LR curves |
| 10 | Generate | Sample text from the best checkpoint |
| 11 | Save | `experiment.json` + W&B artifact |

**Prerequisites:** Run notebooks 2 and 4 first. Install `wandb` and `tqdm`, then `wandb login`.

**Outputs:** `sweep_best_config.json`, `gpt_best.pt`, `experiment.json`, W&B artifact `sweep-best-v1`.

> The **test** split is intentionally not used here — it is reserved for notebook 6.


## 1. Setup

Import libraries and locate the project `data/` folder (works whether you launch Jupyter from the repo root or from `notebooks/`).


In [ ]:
from pathlib import Path
import random
import re
import time
import math
import json
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import wandb
except ImportError as e:
    raise ImportError("Install wandb: pip install wandb") from e

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-darkgrid")


In [ ]:
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate a 'data' directory from cwd: {cwd}")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = DATA_DIR / "artifacts"
CHECKPOINT_DIR = ARTIFACTS_DIR / "model"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = CHECKPOINT_DIR / "gpt_best.pt"
SWEEP_CONFIG_PATH = ARTIFACTS_DIR / "sweep_best_config.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {device}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


## 2. Hyperparameter config

We merge three layers of settings:

1. **`BASE_CONFIG`** — fixed training knobs (batch size, max steps, early stopping, W&B names)
2. **Sweep-best params** — `learning_rate`, `n_layers`, `d_model`, etc. from notebook 4
3. **Derived values** — e.g. `min_lr = 0.1 × learning_rate`

**Where sweep params come from** (controlled by `CONFIG_SOURCE`):

| Value | Behavior |
|-------|----------|
| `"auto"` *(default)* | Use cached `sweep_best_config.json` if present; otherwise fetch from W&B; fall back to `FALLBACK_BEST` |
| `"local"` | Always read cached JSON (no network) |
| `"wandb"` | Always fetch from W&B sweep |


In [ ]:
# "auto" | "local" | "wandb"
CONFIG_SOURCE = "auto"
FORCE_WANDB_REFRESH = False

BASE_CONFIG = {
    "seed": 42,
    "block_size": 128,
    "batch_size": 32,
    "max_steps": 5000,
    "eval_interval": 100,
    "eval_batches": 50,
    "betas": (0.9, 0.95),
    "d_ff": 4,
    "grad_accum_steps": 4,
    "use_amp": True,
    "early_stop_patience_steps": 500,
    "wandb_project": "genre-story-generator",
    "wandb_entity": None,
    "wandb_group": "final-run",
    "wandb_job_type": "final-training",
    "run_name": "sweep-best-final-v1",
}

SWEEP_ID = "kbmij8of"
SWEEP_METRIC = "val/loss"
SWEEP_PARAMS = [
    "learning_rate",
    "n_layers",
    "d_model",
    "n_heads",
    "dropout",
    "weight_decay",
    "warmup_steps",
]

FALLBACK_BEST = {
    "run_id": "zhocttbr",
    "run_name": "earthy-sweep-6",
    "val/loss": 1.4392,
    "learning_rate": 0.00125916,
    "n_layers": 5,
    "d_model": 512,
    "n_heads": 8,
    "dropout": 0.15540136469775573,
    "weight_decay": 0.2,
    "warmup_steps": 200,
}

if device == "cpu":
    BASE_CONFIG["use_amp"] = False


In [ ]:
def _sweep_api_path(sweep_id: str) -> str:
    project = BASE_CONFIG["wandb_project"]
    entity = BASE_CONFIG.get("wandb_entity")
    if entity is None:
        entity = wandb.Api().default_entity
    return f"{entity}/{project}/{sweep_id}"


def fetch_sweep_runs(sweep_id: str) -> pd.DataFrame:
    api = wandb.Api()
    sweep = api.sweep(_sweep_api_path(sweep_id))
    rows = []
    for run in sweep.runs:
        hist = run.history(keys=[SWEEP_METRIC, "val/perplexity"])
        if len(hist) and SWEEP_METRIC in hist.columns and hist[SWEEP_METRIC].notna().any():
            best_idx = hist[SWEEP_METRIC].idxmin()
            best_val = float(hist.loc[best_idx, SWEEP_METRIC])
            best_ppl = (
                float(hist.loc[best_idx, "val/perplexity"])
                if "val/perplexity" in hist.columns and pd.notna(hist.loc[best_idx, "val/perplexity"])
                else math.exp(best_val)
            )
        elif run.summary.get(SWEEP_METRIC) is not None:
            best_val = float(run.summary[SWEEP_METRIC])
            best_ppl = float(run.summary.get("val/perplexity") or math.exp(best_val))
        else:
            continue
        row = {
            "run_id": run.id,
            "run_name": run.name,
            "state": run.state,
            SWEEP_METRIC: best_val,
            "val/perplexity": best_ppl,
            "model/parameters": run.summary.get("model/parameters"),
        }
        for param in SWEEP_PARAMS:
            row[param] = run.config.get(param)
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    sort_cols = [SWEEP_METRIC]
    ascending = [True]
    if df["model/parameters"].notna().any():
        sort_cols.append("model/parameters")
        ascending.append(True)
    return df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)


def _apply_best_row(config: dict, best_row: dict) -> dict:
    for param in SWEEP_PARAMS:
        if param in best_row and best_row[param] is not None:
            val = best_row[param]
            if param == "warmup_steps":
                val = int(val)
            elif param in ("n_layers", "n_heads", "d_model"):
                val = int(val)
            config[param] = val
    peak_lr = float(config["learning_rate"])
    config["min_lr"] = peak_lr * 0.1
    config["sweep_best_val_loss"] = float(
        best_row.get(SWEEP_METRIC, best_row.get("val/loss", float("inf")))
    )
    return config


def _patience_steps(config: dict) -> int:
    return int(
        config.get("early_stop_patience_steps")
        or config.get("early_stop_patience")
        or 500
    )


def _write_sweep_config(sweep_meta: dict, config: dict) -> None:
    sweep_meta["sweep_best_val_loss"] = config["sweep_best_val_loss"]
    sweep_meta["hyperparameters"] = {p: config[p] for p in SWEEP_PARAMS}
    sweep_meta["final_config"] = {
        k: (list(v) if isinstance(v, tuple) else v) for k, v in config.items()
    }
    with open(SWEEP_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(sweep_meta, f, indent=2, default=str)
    print(f"Wrote {SWEEP_CONFIG_PATH}")


def resolve_final_config() -> tuple[dict, dict]:
    sweep_meta = {"sweep_id": SWEEP_ID}
    config = {**BASE_CONFIG}
    config["early_stop_patience_steps"] = _patience_steps(config)

    use_local = False
    if CONFIG_SOURCE == "local":
        use_local = True
    elif CONFIG_SOURCE == "auto" and not FORCE_WANDB_REFRESH and SWEEP_CONFIG_PATH.exists():
        with open(SWEEP_CONFIG_PATH, encoding="utf-8") as f:
            cached = json.load(f)
        if cached.get("final_config"):
            use_local = True

    if use_local:
        with open(SWEEP_CONFIG_PATH, encoding="utf-8") as f:
            cached = json.load(f)
        fc = cached["final_config"]
        config.update(fc)
        config["betas"] = tuple(config.get("betas", (0.9, 0.95)))
        config["early_stop_patience_steps"] = _patience_steps(config)
        if device == "cpu":
            config["use_amp"] = False
        sweep_meta.update({k: cached[k] for k in ("source", "sweep_id", "run_id", "run_name") if k in cached})
        sweep_meta["source"] = cached.get("source", "local")
        sweep_meta["sweep_best_val_loss"] = float(
            cached.get("sweep_best_val_loss", config.get("sweep_best_val_loss", float("inf")))
        )
        config["sweep_best_val_loss"] = sweep_meta["sweep_best_val_loss"]
        sweep_meta["hyperparameters"] = {p: config[p] for p in SWEEP_PARAMS}
        _write_sweep_config(sweep_meta, config)
        return config, sweep_meta

    if CONFIG_SOURCE in ("wandb", "auto"):
        try:
            runs_df = fetch_sweep_runs(SWEEP_ID)
        except Exception as exc:
            print(f"W&B fetch failed ({exc}); using FALLBACK_BEST.")
            runs_df = pd.DataFrame()
        if runs_df.empty:
            best_row = FALLBACK_BEST.copy()
            sweep_meta["source"] = "fallback"
        else:
            best_row = runs_df.iloc[0].to_dict()
            sweep_meta["source"] = "wandb"
            sweep_meta["run_id"] = best_row.get("run_id")
            sweep_meta["run_name"] = best_row.get("run_name")
        config = _apply_best_row(config, best_row)
        _write_sweep_config(sweep_meta, config)
        return config, sweep_meta

    raise ValueError(f"Unknown CONFIG_SOURCE: {CONFIG_SOURCE}")


In [ ]:
FINAL_CONFIG, SWEEP_META = resolve_final_config()

print("--- FINAL_CONFIG ---")
for k in [
    "learning_rate", "min_lr", "warmup_steps", "max_steps",
    "n_layers", "d_model", "n_heads", "dropout", "weight_decay",
    "eval_interval", "early_stop_patience_steps",
]:
    print(f"  {k}: {FINAL_CONFIG.get(k)}")
print(f"  sweep_best_val_loss: {FINAL_CONFIG['sweep_best_val_loss']:.4f}")
print(f"  config source: {SWEEP_META.get('source')}")


## 3. Load data

Load preprocessed token tensors from notebook 2. Each batch is a random window of `block_size` characters; targets are shifted by one position (next-character prediction).


In [ ]:
train_ids_path = ARTIFACTS_DIR / "train_ids.pt"
val_ids_path = ARTIFACTS_DIR / "val_ids.pt"
vocab_path = ARTIFACTS_DIR / "char_vocab.json"

for p in (train_ids_path, val_ids_path, vocab_path):
    assert p.exists(), f"Missing {p}. Run preprocessing notebook first."

train_ids = torch.load(train_ids_path, weights_only=True)
val_ids = torch.load(val_ids_path, weights_only=True)

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

vocab_size = vocab_payload["vocab_size"]
stoi = vocab_payload["stoi"]
itos = {int(k): v for k, v in vocab_payload["itos"].items()}

print(f"train_ids: {train_ids.shape} | val_ids: {val_ids.shape} | vocab: {vocab_size}")


In [ ]:
# Seeded later in section 8; declared here so get_batch can reference them.
train_batch_generator = None
eval_batch_generator = None


def encode(text: str):
    return [stoi[ch] for ch in text]


def decode(indices):
    return "".join(itos[int(i)] for i in indices)


def get_batch(
    split: str,
    batch_size: int,
    block_size: int,
    device: str = device,
    generator: torch.Generator | None = None,
):
    data = train_ids if split == "train" else val_ids
    if len(data) <= block_size:
        raise ValueError(f"{split} split too short for block_size={block_size}")
    max_start = len(data) - block_size - 1
    gen = generator if generator is not None else train_batch_generator
    ix = torch.randint(0, max_start, (batch_size,), generator=gen)
    rows = ix.unsqueeze(1) + torch.arange(block_size)
    x = data[rows]
    y = data[rows + 1]
    return x.to(device), y.to(device)


## 4. Model architecture

Decoder-only GPT (same family as notebook 4). Pre-norm transformer blocks with causal self-attention and a character-level output head.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self._causal_cache = None

    def _causal_mask(self, size: int, device: torch.device):
        if self._causal_cache is None or self._causal_cache.size(-1) < size:
            self._causal_cache = torch.tril(torch.ones(size, size, device=device)).view(1, 1, size, size)
        return self._causal_cache[:, :, :size, :size]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = reshape_heads(q), reshape_heads(k), reshape_heads(v)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self._causal_mask(T, x.device)
        att = att.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff_mult: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff_mult * d_model),
            nn.GELU(),
            nn.Linear(d_ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, config: dict):
        super().__init__()
        d_model = config["d_model"]
        n_heads = config["n_heads"]
        n_layers = config["n_layers"]
        dropout = config["dropout"]
        block_size = config["block_size"]
        d_ff_mult = config.get("d_ff", 4)
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff_mult, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)
        x = self.drop(self.token_embedding(idx) + self.pos_embedding(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
    ):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k > 0:
                k = min(top_k, logits.size(-1))
                v, _ = torch.topk(logits, k, dim=-1)
                logits = logits.masked_fill(logits < v[:, [-1]], float("-inf"))
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                remove = cum_probs - F.softmax(sorted_logits, dim=-1) >= top_p
                sorted_logits[remove] = float("-inf")
                logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        return idx


## 5. Learning rate schedule

Linear **warmup** from 0 → peak LR over `warmup_steps`, then **cosine decay** to `min_lr` (= 10% of peak) by `max_steps`.


In [ ]:
try:
    _LRSchedulerBase = torch.optim.lr_scheduler.LRScheduler
except AttributeError:
    _LRSchedulerBase = torch.optim.lr_scheduler._LRScheduler


class CosineWarmupScheduler(_LRSchedulerBase):
    def __init__(self, optimizer, warmup_steps: int, max_steps: int, min_lr: float = 0.0, last_epoch: int = -1):
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch + 1
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.max_steps - self.warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        return [self.min_lr + (base_lr - self.min_lr) * scale for base_lr in self.base_lrs]


def preview_lr_schedule(config: dict, steps: list[int]):
    opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=config["learning_rate"])
    sched = CosineWarmupScheduler(
        opt,
        warmup_steps=config["warmup_steps"],
        max_steps=config["max_steps"],
        min_lr=config["min_lr"],
    )
    print("LR schedule preview:")
    for target in steps:
        while sched.last_epoch + 1 < target:
            opt.step()
            sched.step()
        lr = sched.get_lr()[0]
        print(f"  step {target:5d}: lr = {lr:.6g}")


## 6. Training helpers

Utility functions used by the training loop:

- **`create_model_and_optimizer`** — build GPT + AdamW (weight decay on matrices only) + LR scheduler
- **`estimate_loss`** — average train/val loss over random batches
- **`save_best_checkpoint`** — write `gpt_best.pt` when validation improves


In [ ]:
def autocast_enabled(config: dict) -> bool:
    return bool(config.get("use_amp")) and device == "cuda"


def autocast_device_type() -> str:
    return "cuda" if device == "cuda" else "cpu"


def create_model_and_optimizer(config: dict):
    model = GPTLanguageModel(vocab_size=vocab_size, config=config).to(device)
    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2 and "bias" not in name and "norm" not in name.lower():
            decay_params.append(p)
        else:
            no_decay_params.append(p)
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": config["weight_decay"]},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=config["learning_rate"],
        betas=tuple(config.get("betas", (0.9, 0.95))),
    )
    scheduler = CosineWarmupScheduler(
        optimizer,
        warmup_steps=config["warmup_steps"],
        max_steps=config["max_steps"],
        min_lr=config["min_lr"],
    )
    return model, optimizer, scheduler


@torch.no_grad()
def estimate_loss(
    model: nn.Module,
    config: dict,
    num_batches: int | None = None,
    train_generator: torch.Generator | None = None,
    eval_generator: torch.Generator | None = None,
):
    model.eval()
    num_batches = num_batches or config["eval_batches"]
    out = {}
    for split in ("train", "val"):
        gen = train_generator if split == "train" else eval_generator
        if gen is None:
            gen = train_batch_generator if split == "train" else eval_batch_generator
        losses = []
        for _ in range(num_batches):
            x, y = get_batch(
                split, config["batch_size"], config["block_size"], device, generator=gen
            )
            with torch.amp.autocast(
                device_type=autocast_device_type(), enabled=autocast_enabled(config)
            ):
                _, loss = model(x, y)
            losses.append(loss.item())
        avg = float(np.mean(losses))
        out[split] = {"loss": avg, "ppl": math.exp(avg)}
    model.train()
    return out


def save_best_checkpoint(model: nn.Module, config: dict, path: Path = CKPT_PATH):
    payload = {
        "model": {k: v.cpu().clone() for k, v in model.state_dict().items()},
        "config": copy.deepcopy(config),
    }
    torch.save(payload, path)


## 7. Training loop

`train_final_run` implements the full training cycle:

1. **Step-0 eval** — baseline val loss vs sweep best
2. **Train** — AMP + gradient accumulation + grad clip + LR step each optimizer step
3. **Eval every `eval_interval` steps** — log metrics, save best checkpoint
4. **Early stop** — halt after `early_stop_patience_steps` optimizer steps without val improvement

Returns a dict with `history`, `best_val_loss`, timing, etc.


In [ ]:
def train_final_run(model, optimizer, scheduler, scaler, config: dict, run) -> dict:
    max_steps = config["max_steps"]
    eval_interval = config["eval_interval"]
    grad_accum = config["grad_accum_steps"]
    patience_steps = _patience_steps(config)
    sweep_best_val_loss = config["sweep_best_val_loss"]
    tokens_per_optimizer_step = config["batch_size"] * config["block_size"] * grad_accum

    best_val_loss = float("inf")
    steps_without_improvement = 0
    early_stopped = False
    stopped_step = max_steps
    tokens_per_sec_samples = []

    history = {
        "step": [],
        "train_loss": [],
        "val_loss": [],
        "train_ppl": [],
        "val_ppl": [],
        "lr": [],
    }

    model.train()
    t0 = time.time()

    step0 = estimate_loss(
        model, config, train_generator=train_batch_generator, eval_generator=eval_batch_generator
    )
    print(
        f"Step 0 (init) | val {step0['val']['loss']:.4f} (ppl {step0['val']['ppl']:.1f}) | "
        f"train {step0['train']['loss']:.4f} | sweep best {sweep_best_val_loss:.4f}"
    )
    run.log(
        {
            "val/loss": step0["val"]["loss"],
            "train/loss": step0["train"]["loss"],
            "val/perplexity": step0["val"]["ppl"],
            "train/perplexity": step0["train"]["ppl"],
            "baseline/sweep_best_val_loss": sweep_best_val_loss,
        },
        step=0,
    )

    pbar = tqdm(range(max_steps), desc="Training", unit="step")
    for step in pbar:
        step_start = time.time()
        optimizer.zero_grad(set_to_none=True)
        micro_loss = 0.0
        for _ in range(grad_accum):
            x, y = get_batch(
                "train",
                config["batch_size"],
                config["block_size"],
                device,
                generator=train_batch_generator,
            )
            with torch.amp.autocast(
                device_type=autocast_device_type(), enabled=autocast_enabled(config)
            ):
                _, loss = model(x, y)
                loss = loss / grad_accum
            scaler.scale(loss).backward()
            micro_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step_elapsed = max(time.time() - step_start, 1e-9)
        tokens_per_sec_samples.append(tokens_per_optimizer_step / step_elapsed)

        if (step + 1) % eval_interval == 0 or step == max_steps - 1:
            metrics = estimate_loss(
                model,
                config,
                train_generator=train_batch_generator,
                eval_generator=eval_batch_generator,
            )
            tl, vl = metrics["train"]["loss"], metrics["val"]["loss"]
            lr = optimizer.param_groups[0]["lr"]
            history["step"].append(step + 1)
            history["train_loss"].append(tl)
            history["val_loss"].append(vl)
            history["train_ppl"].append(metrics["train"]["ppl"])
            history["val_ppl"].append(metrics["val"]["ppl"])
            history["lr"].append(lr)

            log_payload = {
                "train/loss": tl,
                "val/loss": vl,
                "val/perplexity": metrics["val"]["ppl"],
                "train/perplexity": metrics["train"]["ppl"],
                "optim/lr": lr,
                "train/micro_loss": micro_loss * grad_accum,
                "train/step": step + 1,
                "early_stop/steps_without_improvement": steps_without_improvement,
            }

            if vl < best_val_loss:
                best_val_loss = vl
                steps_without_improvement = 0
                save_best_checkpoint(model, config)
                log_payload["checkpoint/saved"] = 1
                log_payload["checkpoint/best_val_loss"] = best_val_loss
                msg = (
                    f"NEW BEST val {vl:.4f} (ppl {metrics['val']['ppl']:.1f}) | "
                    f"train {tl:.4f} | saved {CKPT_PATH.name}"
                )
            else:
                steps_without_improvement += eval_interval
                log_payload["checkpoint/saved"] = 0
                msg = (
                    f"val {vl:.4f} (ppl {metrics['val']['ppl']:.1f}) | train {tl:.4f} | "
                    f"no improve {steps_without_improvement}/{patience_steps} opt steps"
                )

            pbar.write(f"  Step {step+1:5d} | {msg}")
            wandb.log(log_payload, step=step + 1)

            if steps_without_improvement >= patience_steps:
                early_stopped = True
                stopped_step = step + 1
                pbar.write(
                    f"Early stopping at step {stopped_step}: "
                    f"no val improvement for {patience_steps} optimizer steps."
                )
                break

    elapsed = time.time() - t0
    return {
        "history": history,
        "best_val_loss": best_val_loss,
        "early_stopped": early_stopped,
        "stopped_step": stopped_step,
        "elapsed": elapsed,
        "tokens_per_sec_mean": float(np.mean(tokens_per_sec_samples)) if tokens_per_sec_samples else 0.0,
        "sweep_best_val_loss": sweep_best_val_loss,
    }


## 8. Run training

This is the only cell that **starts training**. It:

1. Locks random seeds
2. Previews the LR schedule
3. Builds the model and starts a W&B run
4. Calls `train_final_run`
5. Reloads the best checkpoint weights


In [ ]:
set_seed(FINAL_CONFIG["seed"])
train_batch_generator = torch.Generator().manual_seed(FINAL_CONFIG["seed"])
eval_batch_generator = torch.Generator().manual_seed(FINAL_CONFIG["seed"] + 1)

preview_lr_schedule(FINAL_CONFIG, [1, 200, 201, 2500, FINAL_CONFIG["max_steps"]])

model, optimizer, scheduler = create_model_and_optimizer(FINAL_CONFIG)
scaler = torch.amp.GradScaler(enabled=autocast_enabled(FINAL_CONFIG))
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

run = wandb.init(
    project=FINAL_CONFIG["wandb_project"],
    entity=FINAL_CONFIG.get("wandb_entity"),
    group=FINAL_CONFIG["wandb_group"],
    job_type=FINAL_CONFIG["wandb_job_type"],
    name=FINAL_CONFIG.get("run_name"),
    config=FINAL_CONFIG,
)
wandb.log({"model/parameters": n_params})

train_result = train_final_run(model, optimizer, scheduler, scaler, FINAL_CONFIG, run)

history = train_result["history"]
best_val_loss = train_result["best_val_loss"]
early_stopped = train_result["early_stopped"]
stopped_step = train_result["stopped_step"]
elapsed = train_result["elapsed"]
tokens_per_sec_mean = train_result["tokens_per_sec_mean"]
sweep_best_val_loss = train_result["sweep_best_val_loss"]

print(
    f"Training finished in {elapsed:.1f}s | stopped_step={stopped_step} | "
    f"early_stopped={early_stopped} | mean tokens/s={tokens_per_sec_mean:.0f}"
)

if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    print(f"Restored best checkpoint (val/loss={best_val_loss:.4f})")
else:
    print("Warning: no checkpoint saved (val loss never improved). Using last weights.")


## 9. Training curves

Plot loss, perplexity, and learning rate. Compare final best val loss against the sweep baseline.


In [ ]:
PLOT_PATH = CHECKPOINT_DIR / "training_curves.png"

if not history["step"]:
    print("No eval history to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["step"], history["train_loss"], label="train", marker="o", markersize=3)
    axes[0].plot(history["step"], history["val_loss"], label="val", marker="o", markersize=3)
    axes[0].axhline(sweep_best_val_loss, color="gray", ls="--", label="sweep best")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(history["step"], history["train_ppl"], label="train ppl", marker="o", markersize=3)
    axes[1].plot(history["step"], history["val_ppl"], label="val ppl", marker="o", markersize=3)
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("perplexity")
    axes[1].set_title("Perplexity")
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(PLOT_PATH, dpi=120, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 3))
    plt.plot(history["step"], history["lr"], color="C2")
    plt.xlabel("step")
    plt.ylabel("learning rate")
    plt.title("Cosine warmup + decay schedule")
    plt.tight_layout()
    plt.show()

    gap = history["val_loss"][-1] - history["train_loss"][-1]
    delta = best_val_loss - sweep_best_val_loss
    status = "PASS" if best_val_loss <= sweep_best_val_loss else "REGRESSED"
    print(f"--- Final vs sweep ---")
    print(f"  final best val loss:   {best_val_loss:.4f}")
    print(f"  sweep best val loss:   {sweep_best_val_loss:.4f}")
    print(f"  delta (final - sweep): {delta:+.4f}  [{status}]")
    print(f"  last eval gap (val - train): {gap:+.4f}")


## 10. Qualitative generation

Sample text from the restored best checkpoint. Two decoding modes:

- **greedy** — `top_k=1` (most likely next character)
- **creative** — temperature + top-k / top-p sampling


In [ ]:
CHAR_HEADER_RE = re.compile(r"^[A-Z][A-Z\s]+:")


def has_character_header(text: str) -> bool:
    first_line = text.strip().split("\n", 1)[0]
    return bool(CHAR_HEADER_RE.match(first_line))


model.eval()
prompts = {
    "greedy": {"temperature": 1.0, "top_k": 1, "top_p": 1.0},
    "creative": {"temperature": 0.8, "top_k": 40, "top_p": 0.9},
}

generated_samples = {}
table_rows = []
for label, samp_kwargs in prompts.items():
    for prompt_text in ["ROMEO:", "KING:", "JULIET:"]:
        key = f"{label} | {prompt_text}"
        context = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)
        ids = model.generate(context, max_new_tokens=300, **samp_kwargs)
        text = decode(ids[0].tolist())
        generated_samples[key] = text
        table_rows.append([key, has_character_header(text), text[:2000]])

for key, text in generated_samples.items():
    print("=" * 60)
    print(key)
    print(text[:500])
    if len(text) > 500:
        print("...")

wandb.log({
    "samples": wandb.Table(
        columns=["setting", "has_character_header", "text"],
        data=table_rows,
    )
})


## 11. Save artifacts

Write `experiment.json` locally and upload checkpoint + metadata to W&B for notebook 6.


In [ ]:
final_metrics = estimate_loss(
    model,
    FINAL_CONFIG,
    train_generator=train_batch_generator,
    eval_generator=eval_batch_generator,
)

beat_sweep = best_val_loss <= sweep_best_val_loss
generalization_gap = final_metrics["val"]["loss"] - final_metrics["train"]["loss"]

experiment = {
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in FINAL_CONFIG.items()},
    "sweep_meta": SWEEP_META,
    "best_val_loss": best_val_loss,
    "sweep_best_val_loss": sweep_best_val_loss,
    "beat_sweep": beat_sweep,
    "final_metrics": final_metrics,
    "generalization_gap": generalization_gap,
    "stopped_step": stopped_step,
    "early_stopped": early_stopped,
    "training_seconds": elapsed,
    "tokens_per_sec_mean": tokens_per_sec_mean,
    "generated_samples": generated_samples,
    "checkpoint_path": str(CKPT_PATH),
}

exp_path = CHECKPOINT_DIR / "experiment.json"
with open(exp_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, indent=2)

artifact = wandb.Artifact("sweep-best-v1", type="model")
artifact.add_file(str(CKPT_PATH))
artifact.add_file(str(vocab_path))
artifact.add_file(str(SWEEP_CONFIG_PATH))
artifact.add_file(str(exp_path))
if PLOT_PATH.exists():
    artifact.add_file(str(PLOT_PATH))
run.log_artifact(artifact)

run.summary.update({
    "best_val_loss": best_val_loss,
    "sweep_best_val_loss": sweep_best_val_loss,
    "beat_sweep": beat_sweep,
    "stopped_step": stopped_step,
    "early_stopped": early_stopped,
    "tokens_per_sec_mean": tokens_per_sec_mean,
})
run.finish()

print(f"Saved:\n  {CKPT_PATH}\n  {exp_path}")
print(f"beat_sweep={beat_sweep} | generalization_gap={generalization_gap:+.4f}")
